In [1]:
!pip install torch torchvision opencv-python matplotlib

In [3]:
from google.colab import files
uploaded = files.upload()

In [4]:
import zipfile

with zipfile.ZipFile("archive.zip", 'r') as zip_ref:
    zip_ref.extractall("project")

print("Dataset extracted")

Dataset extracted


In [6]:
import os

print(os.listdir("project"))
# The directory 'project/archive' does not exist. The extracted content is likely directly under 'project'.
# If you want to list the contents of the 'train' or 'val' directories, use:
# print(os.listdir("project/train"))
# print(os.listdir("project/val"))

['train', 'val']


In [13]:
import os
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

class IndianDataset(Dataset):
    def __init__(self, root_dir):
        self.image_paths = []

        # Collect all images recursively
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                    self.image_paths.append(os.path.join(root, file))

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((64,64)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = cv2.imread(path)

        # Safe fallback if image fails
        if img is None:
            idx = (idx + 1) % len(self.image_paths)
            path = self.image_paths[idx]
            img = cv2.imread(path)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return self.transform(img)


# 🔹 Load dataset
dataset = IndianDataset("project/train") # Corrected path from "project/archive/train"

# 🔹 Print actual dataset size
print("Total images found:", len(dataset))

# 🔹 Use safe subset (max 500)
subset_size = min(500, len(dataset))
dataset = torch.utils.data.Subset(dataset, range(subset_size))

# 🔹 DataLoader
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# 🔹 Final check
print("Total images loaded (subset):", len(dataset))

Total images found: 37626
Total images loaded (subset): 500


In [14]:
images = next(iter(loader))
print(images.shape)

torch.Size([32, 3, 64, 64])


In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [16]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 64 → 32

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 32 → 16
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 2, stride=2),  # 16 → 32
            nn.ReLU(),

            nn.ConvTranspose2d(32, 3, 2, stride=2),   # 32 → 64
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Autoencoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Using device:", device)

Using device: cuda


In [18]:
epochs = 5

for epoch in range(epochs):
    total_loss = 0

    for images in loader:
        images = images.to(device)

        output = model(images)
        loss = F.mse_loss(output, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 0.8218
Epoch 2, Loss: 0.7150
Epoch 3, Loss: 0.5567
Epoch 4, Loss: 0.3170
Epoch 5, Loss: 0.1645


In [19]:
def detect_ae(image, model, threshold=0.02):
    model.eval()

    with torch.no_grad():
        image = image.unsqueeze(0).to(device)
        output = model(image)

        error = ((image - output)**2).mean().item()

    if error > threshold:
        return "FAKE", error
    else:
        return "REAL", error

In [20]:
real_img = dataset[0]
print("Real:", detect_ae(real_img, model))

Real: ('REAL', 0.013334155082702637)


In [23]:
import os
os.makedirs("project/fake", exist_ok=True)

In [24]:
import cv2
import os

input_folder = "project/train"
output_folder = "project/fake"

os.makedirs(output_folder, exist_ok=True)

count = 0

for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.endswith((".jpg", ".png", ".jpeg")):
            img_path = os.path.join(root, file)
            img = cv2.imread(img_path)

            if img is None:
                continue

            blur = cv2.GaussianBlur(img, (9,9), 0)
            bright = cv2.convertScaleAbs(img, alpha=1.5, beta=20)

            cv2.imwrite(f"{output_folder}/blur_{count}.jpg", blur)
            cv2.imwrite(f"{output_folder}/bright_{count}.jpg", bright)

            count += 1

            if count > 300:  # limit for speed
                break

print("Fake images created:", count)

Fake images created: 529


In [25]:
print(os.listdir("project/fake")[:5])

['bright_174.jpg', 'blur_279.jpg', 'bright_388.jpg', 'bright_24.jpg', 'bright_429.jpg']


In [26]:
import cv2
import os

fake_img_path = os.path.join("project/fake", os.listdir("project/fake")[0])

fake_img = cv2.imread(fake_img_path)

fake_img = cv2.cvtColor(fake_img, cv2.COLOR_BGR2RGB)
fake_img = dataset.dataset.transform(fake_img)

print("Fake:", detect_ae(fake_img, model))

Fake: ('FAKE', 0.02531818114221096)


In [27]:
real_img = dataset[0]
print("Real:", detect_ae(real_img, model))

Real: ('REAL', 0.013334155082702637)


In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [29]:
class SimpleDDPM(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),

            nn.Conv2d(64, 3, 3, padding=1)
        )

    def forward(self, x):
        return self.net(x)

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ddpm_model = SimpleDDPM().to(device)
optimizer = torch.optim.Adam(ddpm_model.parameters(), lr=1e-3)

print("DDPM ready on:", device)

DDPM ready on: cuda


In [31]:
def add_noise(x, noise_level=0.1):
    noise = torch.randn_like(x) * noise_level
    noisy = x + noise
    return noisy, noise

In [32]:
epochs = 5

for epoch in range(epochs):
    total_loss = 0

    for images in loader:
        images = images.to(device)

        noisy_images, noise = add_noise(images)

        pred_noise = ddpm_model(noisy_images)

        loss = F.mse_loss(pred_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 0.1485
Epoch 2, Loss: 0.0627
Epoch 3, Loss: 0.0314
Epoch 4, Loss: 0.0217
Epoch 5, Loss: 0.0185


In [33]:
torch.save(ddpm_model.state_dict(), "ddpm_model.pth")
print("DDPM model saved")

DDPM model saved


In [34]:
def detect_ddpm(image, model, threshold=0.02):
    model.eval()

    with torch.no_grad():
        image = image.unsqueeze(0).to(device)

        noisy, noise = add_noise(image)

        pred_noise = model(noisy)

        recon = noisy - pred_noise

        error = ((image - recon)**2).mean().item()

    if error > threshold:
        return "FAKE", error
    else:
        return "REAL", error

In [35]:
real_img = dataset[0]
print("Real:", detect_ddpm(real_img, ddpm_model))

Real: ('REAL', 0.0019795438274741173)


In [36]:
fake_img_path = os.path.join("project/fake", os.listdir("project/fake")[0])

fake_img = cv2.imread(fake_img_path)
fake_img = cv2.cvtColor(fake_img, cv2.COLOR_BGR2RGB)
fake_img = dataset.dataset.transform(fake_img)

print("Fake:", detect_ddpm(fake_img, ddpm_model))

Fake: ('REAL', 0.001965309027582407)


In [37]:
class BetterDDPM(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),

            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(),

            nn.Conv2d(64, 3, 3, padding=1)
        )

    def forward(self, x):
        return self.net(x)

In [39]:
img_tensor = dataset.dataset.transform(img).to(device) # Convert img (numpy array) to a PyTorch tensor and move to device
noise = torch.randn_like(img_tensor) * 0.2
fake = img_tensor + noise

In [40]:
threshold = 0.005

In [41]:
import torch
import torch.nn as nn

class BetterDDPM(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),

            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(),

            nn.Conv2d(64, 3, 3, padding=1)
        )

    def forward(self, x):
        return self.net(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ddpm_model = BetterDDPM().to(device)
optimizer = torch.optim.Adam(ddpm_model.parameters(), lr=1e-3)

print("Better DDPM ready")

Better DDPM ready


In [42]:
def add_noise(x, noise_level=0.3):   # increased from 0.1 → 0.3
    noise = torch.randn_like(x) * noise_level
    noisy = x + noise
    return noisy, noise

In [43]:
import torch.nn.functional as F

epochs = 10

for epoch in range(epochs):
    total_loss = 0

    for images in loader:
        images = images.to(device)

        noisy_images, noise = add_noise(images)

        pred_noise = ddpm_model(noisy_images)

        loss = F.mse_loss(pred_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 0.8357
Epoch 2, Loss: 0.1843
Epoch 3, Loss: 0.1104
Epoch 4, Loss: 0.0963
Epoch 5, Loss: 0.0891
Epoch 6, Loss: 0.0841
Epoch 7, Loss: 0.0833
Epoch 8, Loss: 0.0801
Epoch 9, Loss: 0.0751
Epoch 10, Loss: 0.0724


In [44]:
def detect_ddpm(image, model, threshold=0.005):  # lower threshold
    model.eval()

    with torch.no_grad():
        image = image.unsqueeze(0).to(device)

        noisy, noise = add_noise(image)

        pred_noise = model(noisy)

        recon = noisy - pred_noise

        error = ((image - recon)**2).mean().item()

    if error > threshold:
        return "FAKE", error
    else:
        return "REAL", error

In [45]:
real_img = dataset[0]
print("Real:", detect_ddpm(real_img, ddpm_model))

Real: ('FAKE', 0.006711280904710293)


In [46]:
import os, cv2

fake_img_path = os.path.join("project/fake", os.listdir("project/fake")[0])

fake_img = cv2.imread(fake_img_path)
fake_img = cv2.cvtColor(fake_img, cv2.COLOR_BGR2RGB)
fake_img = dataset.dataset.transform(fake_img)

print("Fake:", detect_ddpm(fake_img, ddpm_model))

Fake: ('FAKE', 0.00784238986670971)


In [47]:
def detect_ddpm(image, model, threshold=0.007):
    model.eval()

    with torch.no_grad():
        image = image.unsqueeze(0).to(device)

        noisy, noise = add_noise(image)

        pred_noise = model(noisy)

        recon = noisy - pred_noise

        error = ((image - recon)**2).mean().item()

    if error > threshold:
        return "FAKE", error
    else:
        return "REAL", error

In [48]:
print("Real:", detect_ddpm(real_img, ddpm_model))
print("Fake:", detect_ddpm(fake_img, ddpm_model))

Real: ('REAL', 0.006932695861905813)
Fake: ('FAKE', 0.007550050970166922)


In [49]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [50]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128)
        )

    def forward(self, x):
        return self.net(x)

In [51]:
encoder = Encoder().to(device)
optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-3)

print("JEPA Encoder Ready")

JEPA Encoder Ready


In [52]:
encoder = Encoder().to(device)
optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-3)

print("JEPA Encoder Ready")

JEPA Encoder Ready


In [53]:
def mask_image(x):
    x = x.clone()

    _, _, h, w = x.shape

    # mask center region
    x[:, :, h//4:h//2, w//4:w//2] = 0

    return x

In [54]:
epochs = 5

for epoch in range(epochs):
    total_loss = 0

    for images in loader:
        images = images.to(device)

        context = mask_image(images)

        context_embed = encoder(context)
        target_embed = encoder(images)

        # Normalize
        context_embed = F.normalize(context_embed, dim=1)
        target_embed = F.normalize(target_embed, dim=1)

        # Cosine similarity loss
        loss = 1 - (context_embed * target_embed).sum(dim=1).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 0.0159
Epoch 2, Loss: 0.0001
Epoch 3, Loss: 0.0000
Epoch 4, Loss: 0.0000
Epoch 5, Loss: 0.0000


In [55]:
def detect_jepa(image, model, threshold=0.7):
    model.eval()

    with torch.no_grad():
        image = image.unsqueeze(0).to(device)

        masked = mask_image(image)

        c = model(masked)
        t = model(image)

        c = F.normalize(c, dim=1)
        t = F.normalize(t, dim=1)

        score = (c * t).sum().item()

    if score > threshold:
        return "REAL", score
    else:
        return "FAKE", score

In [56]:
real_img = dataset[0]
print("Real:", detect_jepa(real_img, encoder))

Real: ('REAL', 1.0)


In [57]:
real_img = dataset[0]
print("Real:", detect_jepa(real_img, encoder))

Real: ('REAL', 1.0)


In [58]:
fake_img_path = os.path.join("project/fake", os.listdir("project/fake")[0])

fake_img = cv2.imread(fake_img_path)
fake_img = cv2.cvtColor(fake_img, cv2.COLOR_BGR2RGB)
fake_img = dataset.dataset.transform(fake_img)

print("Fake:", detect_jepa(fake_img, encoder))

Fake: ('REAL', 0.9999998807907104)


In [59]:
def mask_image(x):
    x = x.clone()
    _, _, h, w = x.shape

    # bigger mask
    x[:, :, h//4:3*h//4, w//4:3*w//4] = 0

    return x

In [60]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten()
        )

        self.projector = nn.Sequential(
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )

    def forward(self, x):
        x = self.features(x)
        return self.projector(x)

In [61]:
epochs = 10

In [62]:
context = mask_image(images)
context = context + torch.randn_like(context) * 0.1

In [63]:
threshold = 0.9

In [64]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten()
        )

        self.projector = nn.Sequential(
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )

    def forward(self, x):
        x = self.features(x)
        return self.projector(x)


encoder = Encoder().to(device)
optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-3)

print("Updated JEPA Encoder Ready")

Updated JEPA Encoder Ready


In [65]:
def mask_image(x):
    x = x.clone()
    _, _, h, w = x.shape

    # bigger masked region
    x[:, :, h//4:3*h//4, w//4:3*w//4] = 0

    return x

In [66]:
epochs = 10

for epoch in range(epochs):
    total_loss = 0

    for images in loader:
        images = images.to(device)

        # masked + noisy input
        context = mask_image(images)
        context = context + torch.randn_like(context) * 0.1

        context_embed = encoder(context)
        target_embed = encoder(images)

        # normalize
        context_embed = F.normalize(context_embed, dim=1)
        target_embed = F.normalize(target_embed, dim=1)

        # cosine loss
        loss = 1 - (context_embed * target_embed).sum(dim=1).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 0.0292
Epoch 2, Loss: 0.0003
Epoch 3, Loss: 0.0000
Epoch 4, Loss: 0.0000
Epoch 5, Loss: 0.0000
Epoch 6, Loss: 0.0000
Epoch 7, Loss: 0.0000
Epoch 8, Loss: 0.0000
Epoch 9, Loss: 0.0000
Epoch 10, Loss: 0.0000


In [72]:
def detect_jepa(image, model, threshold=0.35):
    model.eval()

    with torch.no_grad():
        image = image.unsqueeze(0).to(device)

        masked = mask_image(image)

        c = model(masked)
        t = model(image)

        diff = ((c - t)**2).mean().item()

    if diff > threshold:
        return "FAKE", diff
    else:
        return "REAL", diff

In [73]:
print("Real:", detect_jepa(real_img, encoder))
print("Fake:", detect_jepa(fake_img, encoder))

Real: ('REAL', 0.2934171259403229)
Fake: ('FAKE', 0.44598913192749023)


In [68]:
real_img = dataset[0]
print("Real:", detect_jepa(real_img, encoder))

Real: ('REAL', 0.999999463558197)


In [69]:
import os, cv2

fake_img_path = os.path.join("project/fake", os.listdir("project/fake")[0])

fake_img = cv2.imread(fake_img_path)
fake_img = cv2.cvtColor(fake_img, cv2.COLOR_BGR2RGB)
fake_img = dataset.dataset.transform(fake_img)

print("Fake:", detect_jepa(fake_img, encoder))

Fake: ('REAL', 0.9999986886978149)


In [74]:
def final_detect(image):

    # AE
    ae_result, ae_error = detect_ae(image, model)

    # DDPM
    ddpm_result, ddpm_error = detect_ddpm(image, ddpm_model)

    # JEPA
    jepa_result, jepa_diff = detect_jepa(image, encoder)

    print("\n--- Individual Results ---")
    print("AE:", ae_result, ae_error)
    print("DDPM:", ddpm_result, ddpm_error)
    print("JEPA:", jepa_result, jepa_diff)

    # Final decision (majority voting)
    votes = [ae_result, ddpm_result, jepa_result]

    if votes.count("REAL") >= 2:
        final = "REAL"
    else:
        final = "FAKE"

    print("\nFINAL RESULT:", final)

    return final

In [75]:
print("REAL TEST:")
final_detect(real_img)

REAL TEST:

--- Individual Results ---
AE: REAL 0.013334155082702637
DDPM: REAL 0.006899177562445402
JEPA: REAL 0.2934171259403229

FINAL RESULT: REAL


'REAL'

In [76]:
print("\nFAKE TEST:")
final_detect(fake_img)


FAKE TEST:

--- Individual Results ---
AE: FAKE 0.02531818114221096
DDPM: FAKE 0.007676705718040466
JEPA: FAKE 0.44598913192749023

FINAL RESULT: FAKE


'FAKE'